**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Variational Inference & Normalizing Flows

The [VAE's](./Representation_Learning.ipynb) loss finally justified: the ELBO derived, its gap identified as a KL, and normalizing flows — exact likelihoods through invertible networks — built and audited against a closed-form density.

## 1. Pre-requisites

[Representation Learning](./Representation_Learning.ipynb) S2, [Information Theory](../Intro_Math/Information_Theory/Information_Theory.ipynb) (KL), [Random Variables](../Intro_Math/Analysis/Random_Variables.ipynb) (change of variables).

In [1]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0); rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 3 — *The ELBO, Derived* (~40 min)
**Goal:** one line of algebra: log-likelihood = ELBO + KL(q‖posterior); verified on a conjugate model.
**Builds on:** [Representation Learning](./Representation_Learning.ipynb) S2. &nbsp; **Feeds into:** Session 2 (flows).

---

## 2. The Bound and Its Gap

💡 **Intuition.** Latent-variable likelihoods need an integral over $z$ — intractable. Multiply and divide by any distribution $q(z)$ inside the log, apply [Jensen](../Intro_Math/Information_Theory/Information_Theory.ipynb), and:
$$\log p(x) = \underbrace{E_q[\log p(x, z) - \log q(z)]}_{\text{ELBO}} + \underbrace{KL(q \,\|\, p(z|x))}_{\ge 0, = \text{the gap}}$$
Maximizing the ELBO over $q$ *is* pushing $q$ toward the true posterior; the bound is tight iff they match. The VAE's loss is exactly this with an encoder network as $q$. On a conjugate Gaussian model the posterior is known — so for once we can *watch* the gap close.

In [2]:
# model: z ~ N(0,1), x|z ~ N(z, 0.5²); observe x=1.2. Posterior is closed-form Gaussian.
x_obs = 1.2
s2_lik = 0.25
post_var = 1/(1 + 1/s2_lik)
post_mu = post_var * (x_obs/s2_lik)
log_px = -0.5*np.log(2*np.pi*(1+s2_lik)) - 0.5*x_obs**2/(1+s2_lik)     # exact evidence

# variational family: N(m, s²); optimize the ELBO by gradient ascent (reparameterized MC)
m = torch.tensor(0.0, requires_grad=True)
log_s = torch.tensor(0.0, requires_grad=True)
opt = torch.optim.Adam([m, log_s], lr=0.05)
for step in range(800):
    eps = torch.randn(256)
    z = m + torch.exp(log_s)*eps
    logp = -0.5*np.log(2*np.pi) - 0.5*z**2            -0.5*np.log(2*np.pi*s2_lik) - 0.5*(x_obs - z)**2/s2_lik
    logq = -0.5*np.log(2*np.pi) - log_s - 0.5*eps**2
    elbo = (logp - logq).mean()
    opt.zero_grad(); (-elbo).backward(); opt.step()

kl_gap = log_px - elbo.item()
print(f"true posterior:  N({post_mu:.4f}, {post_var:.4f})")
print(f"learned q:       N({m.item():.4f}, {torch.exp(2*log_s).item():.4f})")
print(f"log p(x) = {log_px:.4f}   final ELBO = {elbo.item():.4f}   gap = {kl_gap:.5f} → ≈ 0: q reached the posterior")

true posterior:  N(0.9600, 0.2000)
learned q:       N(0.9637, 0.1907)
log p(x) = -1.6065   final ELBO = -1.6053   gap = -0.00123 → ≈ 0: q reached the posterior


---
### 🕐 Session 2 of 3 — *Normalizing Flows* (~40 min)
**Goal:** exact densities through invertible maps: change-of-variables with a learnable Jacobian.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (the trade-space).

---

## 3. Exact Likelihood, No Bound

💡 **Intuition.** Push a Gaussian through an *invertible* network $f$ and the [change-of-variables formula](../Intro_Math/Analysis/Random_Variables.ipynb) gives the exact density: $\log p(x) = \log p_z(f^{-1}(x)) + \log|\det J_{f^{-1}}|$. The engineering is making that determinant cheap: **coupling layers** transform half the coordinates using parameters computed from the other half — triangular Jacobian, determinant = product of scales. Stack and alternate halves: an expressive, exactly-normalized density.

In [3]:
# simpler, correct assembly:
class Flow(nn.Module):
    def __init__(self, n_layers=6):
        super().__init__()
        self.nets = nn.ModuleList([nn.Sequential(nn.Linear(1, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2)) for _ in range(n_layers)])
    def forward(self, x):
        logdet = torch.zeros(len(x))
        for i, net in enumerate(self.nets):
            keep, move = (x[:, :1], x[:, 1:]) if i % 2 == 0 else (x[:, 1:], x[:, :1])
            s, t = net(keep).chunk(2, dim=1)
            s = torch.tanh(s)
            moved = move*torch.exp(s) + t
            x = torch.cat([keep, moved], 1) if i % 2 == 0 else torch.cat([moved, keep], 1)
            logdet += s.squeeze(1)
        return x, logdet

# target: the two-moons distribution (reuse the diffusion workshop's)
def moons(n):
    t_ = rng.uniform(0, np.pi, n)
    top = np.stack([np.cos(t_), np.sin(t_)], 1)
    bot = np.stack([1-np.cos(t_), 0.4-np.sin(t_)], 1)
    X = np.concatenate([top[:n//2], bot[n//2:]]) + 0.06*rng.standard_normal((n, 2))
    return torch.tensor(((X - X.mean(0))/X.std(0)), dtype=torch.float32)

flow = Flow(); opt = torch.optim.Adam(flow.parameters(), lr=1e-3)
Xm = moons(6000)
for step in range(4000):
    idx = torch.randint(0, len(Xm), (512,))
    z, logdet = flow(Xm[idx])
    nll = (0.5*(z**2).sum(1) + np.log(2*np.pi) - logdet).mean()   # exact NLL!
    opt.zero_grad(); nll.backward(); opt.step()
print(f"final exact NLL: {nll.item():.3f} nats  (a standard normal fit would give ≈ {0.5*2*np.log(2*np.pi*np.e):.3f})")

# density heatmap — exactly normalized by construction
g = torch.linspace(-2.6, 2.6, 160)
GX, GY = torch.meshgrid(g, g, indexing="xy")
pts = torch.stack([GX.ravel(), GY.ravel()], 1)
with torch.no_grad():
    z, logdet = flow(pts)
    logp = -0.5*(z**2).sum(1) - np.log(2*np.pi) + logdet
p = logp.exp().reshape(160, 160)
cell = (g[1]-g[0])**2
print(f"∫p ≈ {float(p.sum()*cell):.3f}  (exactly-normalized density: should be ≈ 1)")
plt.figure(figsize=(4.4, 3.6))
plt.contourf(GX, GY, p, levels=30)
plt.scatter(*Xm[:800].T, s=1, c="w", alpha=0.4)
plt.title("flow density: exact log-likelihoods, integral ≈ 1")
plt.tight_layout(); plt.show()

final exact NLL: 1.275 nats  (a standard normal fit would give ≈ 2.838)
∫p ≈ 1.000  (exactly-normalized density: should be ≈ 1)


/tmp/ipykernel_3921173/3580017194.py:48: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 3 of 3 — *The Generative Trade-Space* (~25 min)
**Goal:** VAE vs flow vs diffusion vs GAN: what each buys and what each pays.
**Builds on:** Session 2.

---

## 4. The Family Reunion

| | VAE | Flow | [Diffusion](./Diffusion_Models.ipynb) | GAN |
|---|---|---|---|---|
| Likelihood | bound (ELBO) | **exact** | bound/exact (SDE) | none |
| Sampling | 1 pass | 1 pass | many steps | 1 pass |
| Architecture freedom | full | invertible only | full | full |
| Training stability | good | good | **great** | fragile |
| Latent space | semantic | dimension-preserving | noise schedule | semantic |

💡 **Intuition.** Every generative model juggles three balls — sample quality, likelihood access, sampling speed — and each family drops a different one. Diffusion won the 2020s on quality+stability; flows keep the niche where exact density matters (physics, anomaly detection, [compression](../Intro_Math/Information_Theory/Information_Theory.ipynb)); the ELBO remains the shared grammar.

---
## Where next

- [Diffusion II](./Diffusion_Score_SDE.ipynb) — probability flow: diffusion's flow-shaped face.
- [Representation Learning](./Representation_Learning.ipynb) — the VAE, now with its theory installed.